In [1]:
import pandas as pd
import os

folder = r"C:\Users\itsma\Desktop\sepsiswatch\data\raw"
all_patients = []

for file in os.listdir(folder):
    if file.endswith('.psv'):
        df = pd.read_csv(os.path.join(folder, file), sep='|')
        df['patient_id'] = file.replace('.psv', '')
        all_patients.append(df)

final_data = pd.concat(all_patients, ignore_index=True)

# We also need late_count from funnel analysis
late_detect = []
for pid in final_data[final_data['SepsisLabel'] == 1]['patient_id'].unique():
    patient = final_data[final_data['patient_id'] == pid]
    sepsis_hour = patient[patient['SepsisLabel'] == 1]['ICULOS'].min()
    if sepsis_hour > 24:
        late_detect.append(pid)
late_count = len(late_detect)

print(f"Loaded {final_data['patient_id'].nunique():,} patients")
print(f"Late detected sepsis patients: {late_count}")

Loaded 36,111 patients
Late detected sepsis patients: 1396


In [2]:
#Cost Impact Analysis
# Real published research numbers

avg_sepsis_cost = 22000
late_detect_extra = 14000
avg_icu_day_saved = 2.3
cost_per_icu_day = 4000
mortality_reduction = 0.07
late_patient = 1396

extra_cost_tot = late_patient * late_detect_extra
icu_days_wasted = late_patient * avg_icu_day_saved
icu_cost_wasted = icu_days_wasted * cost_per_icu_day
live_at_risk = int(late_patient * mortality_reduction)

print("=" * 50)
print("Cost Impact of Late Sepsis Detection")
print("=" * 50)
print(f"Late detected patients:                 {late_patient:,}")
print(f"Extra cost per late case:               ${late_detect_extra:,}")
print(f"Total excess cost:                      ${extra_cost_tot:,}")
print(f"Wasted ICU days:                        ${icu_days_wasted:,.1f}")
print(f"Cost of wasted ICU days:                ${icu_cost_wasted:,}")
print(f"Total preventable cost:                 ${extra_cost_tot + icu_cost_wasted:,}")
print(f"Estimated lives at risk:                ${live_at_risk}")
print("=" * 50)

Cost Impact of Late Sepsis Detection
Late detected patients:                 1,396
Extra cost per late case:               $14,000
Total excess cost:                      $19,544,000
Wasted ICU days:                        $3,210.8
Cost of wasted ICU days:                $12,843,199.999999998
Total preventable cost:                 $32,387,200.0
Estimated lives at risk:                $97
